<h1>00 | Data Validation and Cleaning</h1>

In [1]:
import hashlib
from pathlib import Path
import pandas as pd

<h2>1. Configuration and constants</h2>

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "data/raw was not found. Run the notebook inside the project repository."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

NBP_FILE = RAW_DIR / "nbp" / "ceny_mieszkan.xlsx"
NBP_EXPECTED_SHA256 = "841d6322268e7a3941b7e17e916eb9d8fe31ff16c547c6c59eeb801327b03671"

GUS_DIR = RAW_DIR / "gus"
GUS_FILES = {
    "income": GUS_DIR / "WYNA_2497_CREL_20260818171909.csv",
    "transactions": GUS_DIR / "RYNE_3777_CREL_20260818172028.csv",
    "sold_units": GUS_DIR / "RYNE_3789_CREL_20260818172246.csv",
    "sold_area": GUS_DIR / "RYNE_3791_CREL_20260818172324.csv",
    "dwellings_started": GUS_DIR / "PRZE_3822_CREL_20260818172527.csv",
    "dwellings_completed": GUS_DIR / "PRZE_3823_CREL_20260818173141.csv",
    "population": GUS_DIR / "LUDN_2914_CREL_20260818190658.csv",
}

CITY_TERYT = {
    "Białystok": "2061000", "Bydgoszcz": "0461000", "Gdańsk": "2261000",
    "Gdynia": "2262000", "Katowice": "2469000", "Kielce": "2661000",
    "Kraków": "1261000", "Lublin": "0663000", "Łódź": "1061000",
    "Olsztyn": "2862000", "Opole": "1661000", "Poznań": "3064000",
    "Rzeszów": "1863000", "Szczecin": "3262000", "Warszawa": "1465000",
    "Wrocław": "0264000", "Zielona Góra": "0862000",
}

TARGET_TERYT = set(CITY_TERYT.values())
assert len(TARGET_TERYT) == 17

assert all(path.exists() for path in [NBP_FILE, *GUS_FILES.values()])

<h2>2. NBP import and block parsing</h2>

In [3]:
NBP_SHEETS = {"primary": "Rynek pierwotny", "secondary": "Rynek wtórny"}
nbp_excel = pd.ExcelFile(NBP_FILE)

assert set(NBP_SHEETS.values()).issubset(nbp_excel.sheet_names)

nbp_primary_raw = pd.read_excel(NBP_FILE, sheet_name=NBP_SHEETS["primary"], header=None)
nbp_secondary_raw = pd.read_excel(NBP_FILE, sheet_name=NBP_SHEETS["secondary"], header=None)

def parse_nbp_block(df, start, end, period_col=None):
    # Column ranges follow the validated structure of this specific NBP workbook.
    cols = list(range(start, end))

    # The period column is stored separately in the hedonic YoY block.
    if period_col is not None:
        cols = [period_col, *cols]

    # Data starts at row 6. Earlier rows contain worksheet metadata.
    out = df.iloc[6:, cols].copy()

    out.columns = out.iloc[0]

    return (
        out.iloc[1:]
        .rename(columns={"Kwartał": "nbp_period"})
        .dropna(subset=["nbp_period"])
        .reset_index(drop=True)
    )

primary_offer = parse_nbp_block(nbp_primary_raw, 0, 21)
primary_transaction = parse_nbp_block(nbp_primary_raw, 23, 44)
secondary_offer = parse_nbp_block(nbp_secondary_raw, 0, 21)
secondary_transaction = parse_nbp_block(nbp_secondary_raw, 23, 44)
hedonic_qoq = parse_nbp_block(nbp_secondary_raw, 46, 66)
hedonic_yoy = parse_nbp_block(nbp_secondary_raw, 68, 87, 46)

def price_to_long(df, market, price):
    return df.melt(
        id_vars="nbp_period", var_name="city", value_name="price_pln_m2"
    ).assign(market_type=market, price_type=price)

nbp_prices = pd.concat([
    price_to_long(primary_offer, "primary", "offer"),
    price_to_long(primary_transaction, "primary", "transaction"),
    price_to_long(secondary_offer, "secondary", "offer"),
    price_to_long(secondary_transaction, "secondary", "transaction"),
], ignore_index=True)

<h2>3. NBP quality checks</h2>

In [4]:
assert hashlib.sha256(NBP_FILE.read_bytes()).hexdigest() == NBP_EXPECTED_SHA256

NBP_AGGREGATES = {"7 miast", "10 miast", "6 miast bez Warszawy"}
NBP_QUARTER_MAP = {"I": 1, "II": 2, "III": 3, "IV": 4}

def add_nbp_period(df):
    out = df.rename(columns={"nbp_period": "nbp_period_source"}).copy()

    parts = out["nbp_period_source"].astype("string").str.extract(r"^(I|II|III|IV)\s+(\d{4})$")

    assert parts.notna().all().all()

    out["nbp_year"] = parts[1].astype(int)

    out["nbp_quarter"] = parts[0].map(NBP_QUARTER_MAP).astype(int)

    out["nbp_period"] = out["nbp_year"].astype(str) + "Q" + out["nbp_quarter"].astype(str)

    out["nbp_period_order"] = out["nbp_year"] * 10 + out["nbp_quarter"]
    return out

nbp_prices_clean = nbp_prices.copy()

nbp_prices_clean["source_geography"] = nbp_prices_clean["city"].astype("string")

nbp_prices_clean["city"] = nbp_prices_clean["source_geography"].str.replace(r"\*+$", "", regex=True)

assert set(nbp_prices_clean["city"].dropna()) == set(CITY_TERYT) | NBP_AGGREGATES

# Keep only the 17 target cities in the city table and exclude aggregate groups.
nbp_prices_clean = nbp_prices_clean[nbp_prices_clean["city"].isin(CITY_TERYT)].copy()
nbp_prices_clean["price_pln_m2"] = pd.to_numeric(nbp_prices_clean["price_pln_m2"], errors="coerce")
nbp_prices_clean = add_nbp_period(nbp_prices_clean)

# Keep Gdynia observations while flagging known methodology breakpoints.
nbp_prices_clean["methodology_break"] = False
nbp_prices_clean["methodology_note"] = pd.Series(pd.NA, index=nbp_prices_clean.index, dtype="string")

offer_break = (
    nbp_prices_clean["city"].eq("Gdynia")
    & nbp_prices_clean["market_type"].eq("primary")
    & nbp_prices_clean["price_type"].eq("offer")
    & nbp_prices_clean["nbp_period"].eq("2023Q1")
)
transaction_break = (
    nbp_prices_clean["city"].eq("Gdynia")
    & nbp_prices_clean["market_type"].eq("primary")
    & nbp_prices_clean["price_type"].eq("transaction")
    & nbp_prices_clean["nbp_period"].eq("2021Q3")
)

nbp_prices_clean.loc[offer_break, ["methodology_break", "methodology_note"]] = [
    True, "Gdynia primary offer-price weighting methodology changed from 2023Q1."
]
nbp_prices_clean.loc[transaction_break, ["methodology_break", "methodology_note"]] = [
    True, "Gdynia primary transaction-price weighting methodology changed from 2021Q3."
]

expected_periods = {
    f"{y}Q{q}" for y in range(2006, 2027) for q in range(1, 5)
    if not (y == 2006 and q < 3) and not (y == 2026 and q > 2)
}

assert set(nbp_prices_clean["city"]) == set(CITY_TERYT)
assert set(nbp_prices_clean["nbp_period"]) == expected_periods
assert not nbp_prices_clean.duplicated(["city", "nbp_period", "market_type", "price_type"]).any()
assert (nbp_prices_clean["price_pln_m2"].dropna() > 0).all()
assert nbp_prices_clean["methodology_break"].sum() == 2

# Source validation confirms exactly four missing values for Opole on the primary market.
missing_prices = nbp_prices_clean[nbp_prices_clean["price_pln_m2"].isna()]
assert len(missing_prices) == 4
assert missing_prices["city"].eq("Opole").all()
assert missing_prices["market_type"].eq("primary").all()

def hedonic_to_long(df, value_name):
    return df.melt(id_vars="nbp_period", var_name="geography", value_name=value_name)

# QoQ and YoY were stored in separate blocks and are joined by geography and period.
nbp_hedonic_clean = hedonic_to_long(hedonic_qoq, "hedonic_qoq_index").merge(
    hedonic_to_long(hedonic_yoy, "hedonic_yoy_index"),
    on=["nbp_period", "geography"], how="outer", validate="one_to_one"
)

nbp_hedonic_clean = nbp_hedonic_clean[
    ~nbp_hedonic_clean["geography"].isin(NBP_AGGREGATES)
].copy()
nbp_hedonic_clean = add_nbp_period(nbp_hedonic_clean)

for col in ["hedonic_qoq_index", "hedonic_yoy_index"]:
    nbp_hedonic_clean[col] = pd.to_numeric(nbp_hedonic_clean[col], errors="coerce")

# Trójmiasto is a city group. All other records are treated as individual cities.
nbp_hedonic_clean["geography_type"] = nbp_hedonic_clean["geography"].eq("Trójmiasto").map(
    {True: "city_group", False: "city"}
)

expected_hedonic = (set(CITY_TERYT) - {"Gdańsk", "Gdynia"}) | {"Trójmiasto"}
assert set(nbp_hedonic_clean["geography"]) == expected_hedonic
assert set(nbp_hedonic_clean["nbp_period"]) == expected_periods
assert not nbp_hedonic_clean.duplicated(["geography", "nbp_period"]).any()
assert (nbp_hedonic_clean[["hedonic_qoq_index", "hedonic_yoy_index"]].stack() > 0).all()

print("NBP QA passed.")

NBP QA passed.


<h2>4. GUS import and TERYT filtering</h2>

In [5]:
TERYT_TO_CITY = {code: city for city, code in CITY_TERYT.items()}
GUS_COMMON_COLUMNS = {"Kod", "Nazwa", "Rok", "Wartosc", "Jednostka miary", "Atrybut"}

def load_gus(path):
    # Read TERYT codes as strings to preserve leading zeros.
    df = pd.read_csv(path, sep=";", decimal=",", dtype={"Kod": "string"}, encoding="utf-8-sig")

    df = df.loc[:, ~df.columns.str.startswith("Unnamed:")]

    df.columns = df.columns.str.strip()

    assert GUS_COMMON_COLUMNS.issubset(df.columns)

    df = df[df["Kod"].isin(TARGET_TERYT)].copy()

    assert set(df["Kod"].dropna()) == TARGET_TERYT

    df["city"] = df["Kod"].map(TERYT_TO_CITY)

    df["Rok"] = pd.to_numeric(df["Rok"], errors="raise").astype(int)
    return df

gus_data = {name: load_gus(path) for name, path in GUS_FILES.items()}

gus_name_map = (gus_data["income"][["Kod", "Nazwa"]].drop_duplicates().set_index("Kod")["Nazwa"])

dim_city = pd.DataFrame({
    "city_name": list(CITY_TERYT),
    "teryt_code": list(CITY_TERYT.values()),
}).sort_values("city_name").reset_index(drop=True)

dim_city["gus_name"] = dim_city["teryt_code"].map(gus_name_map)

dim_city.insert(0, "city_id", range(1, 18))

assert dim_city[["city_id", "city_name", "teryt_code"]].nunique().eq(17).all()
assert dim_city["gus_name"].notna().all()

<h2>5. GUS attribute handling and transformations</h2>

In [6]:
def clean_gus(df):
    out = df.copy()

    out["attribute"] = out["Atrybut"].fillna("").astype("string").str.strip()

    assert set(out["attribute"]).issubset({"", "x", "n"})

    out["value_raw"] = pd.to_numeric(out["Wartosc"], errors="coerce")
    assert not (out["Wartosc"].notna() & out["value_raw"].isna()).any()

    # Treat GUS attributes x and n as missing values even when the source contains a technical zero.
    out["value"] = out["value_raw"].mask(out["attribute"].isin(["x", "n"]))
    return out

gus_cleaned = {name: clean_gus(df) for name, df in gus_data.items()}

income = gus_cleaned["income"]

assert not income.duplicated(["Kod", "Rok"]).any()

gus_income_clean = (
    income[["Kod", "city", "Rok", "value", "attribute"]]
    .rename(columns={
        "Kod": "teryt_code", "city": "city_name", "Rok": "year",
        "value": "avg_gross_monthly_wage_pln"
    })
    .sort_values(["city_name", "year"])
    .reset_index(drop=True)
)

assert len(gus_income_clean) == 408
assert gus_income_clean["year"].agg(["min", "max"]).tolist() == [2002, 2025]

QUARTER_MAP = {"1 kwartał": 1, "2 kwartał": 2, "3 kwartał": 3, "4 kwartał": 4}

def market_metric(name, value_name, attribute_name):
    out = gus_cleaned[name].copy()
    out["quarter"] = out["Okresy"].map(QUARTER_MAP)

    assert out["quarter"].notna().all()

    assert not out.duplicated(["Kod", "Rok", "quarter"]).any()

    return out[["Kod", "city", "Rok", "quarter", "value", "attribute"]].rename(
        columns={"value": value_name, "attribute": attribute_name}
    )

transactions = market_metric("transactions", "transactions", "transactions_attribute")
sold_units = market_metric("sold_units", "sold_units", "sold_units_attribute")
sold_area = market_metric("sold_area", "sold_area_m2", "sold_area_attribute")

# The three metrics share the same grain and are merged into one city, year, and quarter record.
gus_market_clean = transactions.merge(
    sold_units, on=["Kod", "city", "Rok", "quarter"], validate="one_to_one"
).merge(
    sold_area, on=["Kod", "city", "Rok", "quarter"], validate="one_to_one"
)

gus_market_clean["avg_sold_area_m2"] = (
    gus_market_clean["sold_area_m2"]
    / gus_market_clean["sold_units"].where(gus_market_clean["sold_units"].gt(0))
)

gus_market_clean["calendar_quarter"] = (
    gus_market_clean["Rok"].astype(str) + "Q" + gus_market_clean["quarter"].astype(str)
)

unavailable_2025 = gus_market_clean[
    ["transactions_attribute", "sold_units_attribute", "sold_area_attribute"]
].eq("n").all(axis=1)

# 68 equals 17 cities times 4 quarters. GUS marks all 2025 observations as unavailable.
assert unavailable_2025.sum() == 68
assert gus_market_clean.loc[unavailable_2025, "Rok"].eq(2025).all()

gus_market_clean = (
    gus_market_clean[~unavailable_2025]
    .rename(columns={"Kod": "teryt_code", "city": "city_name", "Rok": "year"})
    .sort_values(["city_name", "year", "quarter"])
    .reset_index(drop=True)
)

assert len(gus_market_clean) == 1020
assert not gus_market_clean.duplicated(["teryt_code", "year", "quarter"]).any()

assert (gus_market_clean[["transactions", "sold_units", "sold_area_m2", "avg_sold_area_m2"]].stack() >= 0).all()

In [7]:
# GUS stores this series cumulatively. For example, styczeń-czerwiec means January through June.
SUPPLY_LABELS = [
    "styczeń", "styczeń-luty", "styczeń-marzec", "styczeń-kwiecień",
    "styczeń-maj", "styczeń-czerwiec", "styczeń-lipiec", "styczeń-sierpień",
    "styczeń-wrzesień", "styczeń-październik", "styczeń-listopad", "styczeń-grudzień",
]

SUPPLY_PERIOD_MAP = {label: month for month, label in enumerate(SUPPLY_LABELS, start=1)}

def prepare_supply(name, supply_type):
    out = gus_cleaned[name].copy()
    out["month"] = out["Okresy"].map(SUPPLY_PERIOD_MAP)

    assert out["month"].notna().all()
    assert not out.duplicated(["Kod", "Rok", "month"]).any()

    out = out.sort_values(["Kod", "Rok", "month"]).copy()
    out["cumulative_value"] = out["value"]

    previous = out.groupby(["Kod", "Rok"])["cumulative_value"].shift()

    out["period_value"] = out["cumulative_value"] - previous

    out.loc[out["month"].eq(1), "period_value"] = out.loc[out["month"].eq(1), "cumulative_value"]

    # A negative difference indicates a cumulative series issue and flags the entire city and year.
    issue = out["period_value"].lt(0).groupby([out["Kod"], out["Rok"]]).transform("any")
    out["period_derivation_issue"] = issue

    # When the cumulative series is invalid, monthly values remain missing.
    out["period_value"] = out["period_value"].mask(issue)

    out["months_available"] = out.groupby(["Kod", "Rok"])["cumulative_value"].transform("count")
    out["is_full_year"] = out["months_available"].eq(12)
    out["supply_type"] = supply_type
    out = out[out["cumulative_value"].notna()]

    return out[
        "Kod city Rok month cumulative_value period_value attribute months_available is_full_year period_derivation_issue supply_type".split()
    ]

gus_supply_clean = pd.concat([
    prepare_supply("dwellings_started", "started"),
    prepare_supply("dwellings_completed", "completed"),
], ignore_index=True)

gus_supply_clean = (
    gus_supply_clean
    .rename(columns={"Kod": "teryt_code", "city": "city_name", "Rok": "year"})
    .sort_values(["supply_type", "city_name", "year", "month"])
    .reset_index(drop=True)
)

assert not gus_supply_clean.duplicated(["teryt_code", "year", "month", "supply_type"]).any()
assert (gus_supply_clean["cumulative_value"] >= 0).all()
assert (gus_supply_clean["period_value"].dropna() >= 0).all()

# Only 2026 should be incomplete because the source contains six months for that year.
partial = gus_supply_clean.loc[~gus_supply_clean["is_full_year"], ["year", "months_available"]].drop_duplicates()
assert set(partial["year"]) == {2026}
assert partial["months_available"].eq(6).all()

# The validated source exception is Gdynia 2005 for dwellings started.
issues = gus_supply_clean.loc[
    gus_supply_clean["period_derivation_issue"], ["teryt_code", "year", "supply_type"]
].drop_duplicates()
assert set(map(tuple, issues.to_numpy())) == {("2262000", 2005, "started")}

In [8]:
population = gus_cleaned["population"]

# P2914 contains multiple population definitions, so the project specific definition is selected explicitly.
population = population[
    population["Zakres terytorialny"].eq("miasta na prawach powiatu")
    & population["Miejsce zamieszkania / zameldowania"].eq("miejsce zamieszkania")
    & population["Płeć"].eq("ogółem")
].copy()

population["reference_date"] = population["Stan na dzień"].map({
    "stan na 30 czerwca": "mid_year",
    "stan na 31 grudnia": "year_end",
})

assert population["reference_date"].notna().all()

assert not population.duplicated(["Kod", "Rok", "reference_date"]).any()

gus_population_clean = (
    population[["Kod", "city", "Rok", "reference_date", "value", "attribute"]]
    .rename(columns={
        "Kod": "teryt_code", "city": "city_name",
        "Rok": "year", "value": "population"
    })
    .sort_values(["city_name", "year", "reference_date"])
    .reset_index(drop=True)
)

assert len(gus_population_clean) == 1054
assert gus_population_clean["year"].agg(["min", "max"]).tolist() == [1995, 2025]
assert (gus_population_clean["population"].dropna() > 0).all()

<h2>6. Clean outputs and reconciliation checkpoints</h2>

In [9]:
city_key = dim_city[["city_id", "city_name", "teryt_code"]]

# NBP has no TERYT code, so city_id is added through the controlled city name mapping.
nbp_prices_clean = nbp_prices_clean.rename(columns={"city": "city_name"}).merge(
    city_key, on="city_name", how="left", validate="many_to_one"
)

def attach_city_id(df):
    # GUS tables contain TERYT, so city_id is added using the more stable teryt_code key.
    return df.merge(
        city_key[["city_id", "teryt_code"]],
        on="teryt_code", how="left", validate="many_to_one"
    )

gus_income_clean = attach_city_id(gus_income_clean)
gus_market_clean = attach_city_id(gus_market_clean)
gus_supply_clean = attach_city_id(gus_supply_clean)
gus_population_clean = attach_city_id(gus_population_clean)

PRICE_COLS = "city_id teryt_code city_name nbp_period nbp_year nbp_quarter nbp_period_order market_type price_type price_pln_m2 methodology_break methodology_note source_geography nbp_period_source".split()
HEDONIC_COLS = "geography geography_type nbp_period nbp_year nbp_quarter nbp_period_order hedonic_qoq_index hedonic_yoy_index nbp_period_source".split()
INCOME_COLS = "city_id teryt_code city_name year avg_gross_monthly_wage_pln attribute".split()
MARKET_COLS = "city_id teryt_code city_name year quarter calendar_quarter transactions transactions_attribute sold_units sold_units_attribute sold_area_m2 sold_area_attribute avg_sold_area_m2".split()
SUPPLY_COLS = "city_id teryt_code city_name year month supply_type cumulative_value period_value attribute months_available is_full_year period_derivation_issue".split()
POPULATION_COLS = "city_id teryt_code city_name year reference_date population attribute".split()

outputs = {
    "dim_city.csv": dim_city,
    "nbp_prices_clean.csv": nbp_prices_clean[PRICE_COLS],
    "nbp_hedonic_clean.csv": nbp_hedonic_clean[HEDONIC_COLS],
    "gus_income_clean.csv": gus_income_clean[INCOME_COLS],
    "gus_market_clean.csv": gus_market_clean[MARKET_COLS],
    "gus_supply_clean.csv": gus_supply_clean[SUPPLY_COLS],
    "gus_population_clean.csv": gus_population_clean[POPULATION_COLS],
}

# Define the grain of each table as the columns that must be unique together.
keys = {
    "dim_city.csv": ["city_id"],
    "nbp_prices_clean.csv": ["city_id", "nbp_period", "market_type", "price_type"],
    "nbp_hedonic_clean.csv": ["geography", "nbp_period"],
    "gus_income_clean.csv": ["city_id", "year"],
    "gus_market_clean.csv": ["city_id", "year", "quarter"],
    "gus_supply_clean.csv": ["city_id", "year", "month", "supply_type"],
    "gus_population_clean.csv": ["city_id", "year", "reference_date"],
}

assert set(outputs["nbp_prices_clean.csv"]["city_id"]) == set(dim_city["city_id"])

for name in ["gus_income_clean.csv", "gus_market_clean.csv", "gus_supply_clean.csv", "gus_population_clean.csv"]:
    assert set(outputs[name]["teryt_code"]) == TARGET_TERYT

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
reconciliation = []

for filename, df in outputs.items():
    duplicates = df.duplicated(keys[filename]).sum()
    assert duplicates == 0

    df.to_csv(PROCESSED_DIR / filename, index=False, encoding="utf-8-sig")

    reconciliation.append([filename.removesuffix(".csv"), len(df), duplicates])

display(pd.DataFrame(reconciliation, columns=["table", "rows", "key_duplicates"]))
print("Clean outputs written to: data/processed")

,table,rows,key_duplicates
0,dim_city,17,0
1,nbp_prices_clean,5440,0
2,nbp_hedonic_clean,1280,0
3,gus_income_clean,408,0
4,gus_market_clean,1020,0
5,gus_supply_clean,8772,0
6,gus_population_clean,1054,0


Clean outputs written to: data/processed


## Read back quality check

After export, the files are loaded from disk again to validate the full `DataFrame → CSV → read_csv` boundary.

In [10]:
READ_DTYPES = {
    "dim_city.csv": {"teryt_code": "string"},
    "nbp_prices_clean.csv": {"teryt_code": "string"},
    "gus_income_clean.csv": {"teryt_code": "string"},
    "gus_market_clean.csv": {"teryt_code": "string"},
    "gus_supply_clean.csv": {"teryt_code": "string"},
    "gus_population_clean.csv": {"teryt_code": "string"},
}

reloaded_outputs = {}

for filename, expected_df in outputs.items():
    dtype = READ_DTYPES.get(filename)
    disk_df = pd.read_csv(PROCESSED_DIR / filename, dtype=dtype)
    reloaded_outputs[filename] = disk_df

    assert len(disk_df) == len(expected_df)
    assert disk_df.duplicated(keys[filename]).sum() == 0

dim_city_disk = reloaded_outputs["dim_city.csv"]
assert dim_city_disk["teryt_code"].str.fullmatch(r"\d{7}").all()
assert set(dim_city_disk["teryt_code"]) == TARGET_TERYT

for filename in [
    "gus_income_clean.csv",
    "gus_market_clean.csv",
    "gus_supply_clean.csv",
    "gus_population_clean.csv",
]:
    assert set(reloaded_outputs[filename]["teryt_code"]) == TARGET_TERYT

print("Re-read QA passed.")

Re-read QA passed.
